# SGCRL Four Rooms — JAX-Accelerated
# Reimplements tabular_maze_absorbing.ipynb with all episode collection,
# replay buffer, and action selection as JIT-compiled JAX functions.
# The only host/device transfer is for periodic visualization.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"  # Use GPU 1 only

import functools
import time

import jax
import jax.numpy as jnp
import matplotlib.animation as animation
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
from IPython.display import HTML, display

print(f"JAX devices: {jax.devices()}")
print(f"JAX backend: {jax.default_backend()}")

plt.ion()
%matplotlib inline

## Environment Setup + Precomputed Transition Tables

In [ ]:
# Configure the Four Rooms Maze
HEIGHT, WIDTH = 10, 10
GOAL_COORD = (9, 9)
NUM_ACTIONS = 5  # stay, down, up, right, left
A_TO_DELTA = np.array([[0, 0], [1, 0], [-1, 0], [0, 1], [0, -1]])

# Build walls
walls = np.zeros((HEIGHT, WIDTH), dtype=int)
DOOR_LEN = 2
walls[HEIGHT // 2, :] = 1
doors_h = np.concatenate(
    [WIDTH // 4 + np.arange(DOOR_LEN), WIDTH * 3 // 4 + np.arange(DOOR_LEN)]
)
walls[HEIGHT // 2, doors_h] = 0
walls[:, WIDTH // 2] = 1
doors_v = np.concatenate(
    [HEIGHT // 4 + np.arange(DOOR_LEN), HEIGHT * 3 // 4 + np.arange(DOOR_LEN)]
)
walls[doors_v, WIDTH // 2] = 0

NUM_STATES = HEIGHT * WIDTH
START_STATE = np.ravel_multi_index((0, 0), (HEIGHT, WIDTH))
GOAL_STATE = np.ravel_multi_index(GOAL_COORD, (HEIGHT, WIDTH))

# Absorbing region: rows 0-4, cols 6-9
ABSORBING_REGION = set()
for r in range(0, 5):
    for c in range(6, 10):
        if walls[r, c] == 0:
            ABSORBING_REGION.add(np.ravel_multi_index((r, c), (HEIGHT, WIDTH)))

# Build absorbing boundary contour for visualization.
_absorbing_grid = np.zeros((HEIGHT, WIDTH), dtype=bool)
for s in ABSORBING_REGION:
    r, c = np.unravel_index(s, (HEIGHT, WIDTH))
    _absorbing_grid[r, c] = True

def _build_absorbing_boundary():
    """Return list of (xs, ys) line segments tracing the absorbing region border."""
    segments = []
    for r in range(HEIGHT):
        for c in range(WIDTH):
            if not _absorbing_grid[r, c]:
                continue
            # Cell (r,c) spans x=[c-0.5, c+0.5], y=[r-0.5, r+0.5] with origin="lower"
            if r == 0 or not _absorbing_grid[r - 1, c]:
                segments.append(([c - 0.5, c + 0.5], [r - 0.5, r - 0.5]))
            if r == HEIGHT - 1 or not _absorbing_grid[r + 1, c]:
                segments.append(([c - 0.5, c + 0.5], [r + 0.5, r + 0.5]))
            if c == 0 or not _absorbing_grid[r, c - 1]:
                segments.append(([c - 0.5, c - 0.5], [r - 0.5, r + 0.5]))
            if c == WIDTH - 1 or not _absorbing_grid[r, c + 1]:
                segments.append(([c + 0.5, c + 0.5], [r - 0.5, r + 0.5]))
    return segments

absorbing_boundary = _build_absorbing_boundary()

# ── Precomputed transition tables ──

def _build_transition_table(walls, absorbing_states=None):
    """Build (NUM_STATES, NUM_ACTIONS) int32 transition table."""
    T = np.zeros((NUM_STATES, NUM_ACTIONS), dtype=np.int32)
    for s in range(NUM_STATES):
        for a in range(NUM_ACTIONS):
            di, dj = A_TO_DELTA[a]
            i, j = np.unravel_index(s, (HEIGHT, WIDTH))
            ni, nj = i + di, j + dj
            if 0 <= ni < HEIGHT and 0 <= nj < WIDTH and walls[ni, nj] == 0:
                T[s, a] = np.ravel_multi_index((ni, nj), (HEIGHT, WIDTH))
            else:
                T[s, a] = s
    if absorbing_states is not None:
        for s in absorbing_states:
            T[s, :] = s
    return T

def _build_near_goal_table():
    """(NUM_STATES,) bool: True if within Manhattan distance 1 of goal."""
    gi, gj = np.unravel_index(GOAL_STATE, (HEIGHT, WIDTH))
    near = np.zeros(NUM_STATES, dtype=bool)
    for s in range(NUM_STATES):
        si, sj = np.unravel_index(s, (HEIGHT, WIDTH))
        near[s] = abs(si - gi) + abs(sj - gj) <= 1
    return near

T_baseline = _build_transition_table(walls)
T_absorbing = _build_transition_table(walls, absorbing_states=ABSORBING_REGION)
near_goal = _build_near_goal_table()

T_baseline_jax = jnp.array(T_baseline)
T_absorbing_jax = jnp.array(T_absorbing)
near_goal_jax = jnp.array(near_goal)

# ── Verify against reference step function ──

def _step_py(state, action):
    di, dj = A_TO_DELTA[action]
    i, j = np.unravel_index(state, (HEIGHT, WIDTH))
    ni, nj = i + di, j + dj
    if 0 <= ni < HEIGHT and 0 <= nj < WIDTH and walls[ni, nj] == 0:
        return np.ravel_multi_index((ni, nj), (HEIGHT, WIDTH))
    return state

for s in range(NUM_STATES):
    for a in range(NUM_ACTIONS):
        assert T_baseline[s, a] == _step_py(s, a)
        expected = s if s in ABSORBING_REGION else _step_py(s, a)
        assert T_absorbing[s, a] == expected

print(f"Grid: {HEIGHT}x{WIDTH}, States: {NUM_STATES}, Actions: {NUM_ACTIONS}")
print(f"Start: {START_STATE}, Goal: {GOAL_STATE}")
print(f"Absorbing region: {len(ABSORBING_REGION)} cells")
print("Transition tables verified.")

## JIT-Compiled Functions

In [ ]:
@functools.partial(jax.jit, static_argnames=("max_steps",))
def collect_episode_jax(psi, T, goal, start, entropy_coeff, key, max_steps):
    """Collect one episode on device using softmax policy.

    At each step, computes similarity of all successor states to goal,
    applies softmax (via categorical), and samples an action.

    Returns:
        trajectory: (max_steps + 1,) int32 array of visited states.
    """
    goal_vec = psi[goal]

    def step_fn(carry, _):
        state, k = carry
        k, subkey = jax.random.split(k)
        next_states = T[state]               # (NUM_ACTIONS,)
        sims = psi[next_states] @ goal_vec   # (NUM_ACTIONS,)
        logits = sims / entropy_coeff
        action = jax.random.categorical(subkey, logits)
        ns = next_states[action]
        return (ns, k), ns

    (_, _), traj_steps = jax.lax.scan(step_fn, (start, key), None, length=max_steps)
    return jnp.concatenate([jnp.array([start]), traj_steps])


@functools.partial(jax.jit, static_argnames=("max_steps",))
def eval_episode_jax(psi, T, goal, start, max_steps):
    """Greedy evaluation episode (argmax, no sampling)."""
    goal_vec = psi[goal]

    def step_fn(state, _):
        next_states = T[state]
        sims = psi[next_states] @ goal_vec
        action = jnp.argmax(sims)
        ns = next_states[action]
        return ns, ns

    _, traj_steps = jax.lax.scan(step_fn, start, None, length=max_steps)
    return jnp.concatenate([jnp.array([start]), traj_steps])


@functools.partial(jax.jit, donate_argnums=(0,))
def store_trajectory(buf, write_ptr, trajectory):
    """Write trajectory into circular replay buffer (donates old buf)."""
    return buf.at[write_ptr].set(trajectory)


@functools.partial(jax.jit, static_argnames=("batch_size", "traj_len"))
def sample_batch(replay_buf, num_valid, key, batch_size, gamma, traj_len):
    """Sample (s, s') pairs with geometric future sampling via Gumbel-max."""
    k1, k2, k3 = jax.random.split(key, 3)

    traj_ids = jax.random.randint(k1, (batch_size,), 0, num_valid)
    i_idx = jax.random.randint(k2, (batch_size,), 0, traj_len - 1)

    # Geometric offset via Gumbel-max trick
    max_offset = traj_len - 1
    offsets_range = jnp.arange(max_offset)
    log_weights = offsets_range * jnp.log(gamma)
    remaining = traj_len - i_idx  # (batch_size,)
    mask = offsets_range[None, :] < remaining[:, None]
    gumbel = jax.random.gumbel(k3, (batch_size, max_offset))
    masked_logits = jnp.where(mask, log_weights[None, :] + gumbel, -1e9)
    sampled_offsets = jnp.argmax(masked_logits, axis=1)

    j_idx = i_idx + sampled_offsets
    return replay_buf[traj_ids, i_idx], replay_buf[traj_ids, j_idx]


@jax.jit
def contrastive_update(psi, s_batch, sp_batch, lr):
    """Vectorised CPC contrastive update (same math as original)."""
    psi_s = psi[s_batch]
    psi_p = psi[sp_batch]
    B = psi_s.shape[0]

    dots = psi_s @ psi_p.T
    dots = dots - dots.max(axis=0, keepdims=True)
    exp_logits = jnp.exp(dots)
    P = exp_logits / exp_logits.sum(axis=0, keepdims=True)

    diag_P = jnp.diag(P)
    nll = -jnp.mean(jnp.log(diag_P + 1e-12))

    coeff = jnp.eye(B) - P
    anchor_update = lr * (coeff @ psi_p)
    expected_anchor = P.T @ psi_s
    pos_update = lr * (psi_s - expected_anchor)

    new_psi = psi.at[s_batch].add(anchor_update)
    new_psi = new_psi.at[sp_batch].add(pos_update)

    norms = jnp.linalg.norm(new_psi, axis=1, keepdims=True) + 1e-8
    new_psi = new_psi / norms
    return new_psi, nll

print("JIT functions defined.")

## Training Loop

In [ ]:
def init_psi(num_states, rep_dim, seed):
    """Random unit-normalised initial representations."""
    rng = np.random.RandomState(seed)
    psi = rng.randn(num_states, rep_dim).astype(np.float32) * 0.1
    psi /= np.linalg.norm(psi, axis=1, keepdims=True) + 1e-8
    return psi


def train(T_jax, config, psi_init_np):
    """JAX-accelerated training loop.

    Args:
        T_jax: (NUM_STATES, NUM_ACTIONS) transition table on device.
        config: dict with hyperparameters.
        psi_init_np: (NUM_STATES, rep_dim) numpy initial representations.

    Returns:
        psi_jax: final representations on device.
        metrics: dict of lists.
    """
    max_steps = config["max_steps"]
    traj_len = max_steps + 1
    replay_capacity = config["replay_capacity"]
    batch_size = config["batch_size"]
    gamma = config["gamma"]
    lr = config["lr_psi"]
    entropy_coeff = config["entropy_coeff"]
    episodes_per_upd = config["episodes_per_upd"]
    plot_freq = config["plot_freq"]
    max_episodes = config["max_episodes"]

    psi_jax = jnp.array(psi_init_np)
    replay_buf = jnp.zeros((replay_capacity, traj_len), dtype=jnp.int32)
    write_ptr = 0
    num_valid = 0

    key = jax.random.PRNGKey(config["seed"])

    success_list = []
    loss_history = []
    eval_success_list = []
    similarity_history = []

    # Warmup JIT (first call triggers compilation)
    key, warmup_key = jax.random.split(key)
    _ = collect_episode_jax(
        psi_jax, T_jax, GOAL_STATE, START_STATE,
        entropy_coeff, warmup_key, max_steps,
    )
    _ = eval_episode_jax(psi_jax, T_jax, GOAL_STATE, START_STATE, max_steps)
    print("JIT warmup complete.")

    t0 = time.perf_counter()

    for ep in tqdm(range(max_episodes), desc="Training"):
        # ── Collect episode (JIT) ──
        key, ep_key = jax.random.split(key)
        traj = collect_episode_jax(
            psi_jax, T_jax, GOAL_STATE, START_STATE,
            entropy_coeff, ep_key, max_steps,
        )

        # Success: did the trajectory visit goal or near-goal?
        is_success = jnp.any(
            near_goal_jax[traj[1:]] | (traj[1:] == GOAL_STATE)
        )
        success_list.append(float(is_success))

        # ── Store in replay buffer ──
        replay_buf = store_trajectory(replay_buf, write_ptr, traj)
        write_ptr = (write_ptr + 1) % replay_capacity
        num_valid = min(num_valid + 1, replay_capacity)

        # ── Contrastive update (JIT) ──
        if ep % episodes_per_upd == 0 and num_valid >= 2:
            key, sample_key = jax.random.split(key)
            s_batch, sp_batch = sample_batch(
                replay_buf, num_valid, sample_key,
                batch_size, gamma, traj_len,
            )
            psi_jax, nll = contrastive_update(
                psi_jax, s_batch, sp_batch, lr,
            )
            loss_history.append(float(nll))

        # ── Periodic eval + similarity snapshot ──
        if ep % plot_freq == 0 or ep == 0:
            eval_traj = eval_episode_jax(
                psi_jax, T_jax, GOAL_STATE, START_STATE, max_steps,
            )
            eval_traj_np = np.array(eval_traj)
            eval_success = bool(
                jnp.any(near_goal_jax[eval_traj] | (eval_traj == GOAL_STATE))
            )
            eval_success_list.append(eval_success)

            # Pull psi to host only for visualization
            psi_np = np.array(psi_jax)
            goal_vec = psi_np[GOAL_STATE]
            gnorm = np.linalg.norm(goal_vec) + 1e-8
            sim = (psi_np @ goal_vec) / (
                np.linalg.norm(psi_np, axis=1) * gnorm + 1e-8
            )
            similarity_history.append((ep, sim, eval_traj_np.tolist()))

    elapsed = time.perf_counter() - t0
    print(
        f"Done: {max_episodes} episodes in {elapsed:.1f}s "
        f"({max_episodes / elapsed:.0f} ep/s)"
    )

    return psi_jax, {
        "success_list": success_list,
        "loss_history": loss_history,
        "eval_success_list": eval_success_list,
        "similarity_history": similarity_history,
    }

## Configuration and Training — Baseline

In [ ]:
config = {
    "rep_dim": 16,
    "episodes_per_upd": 1,
    "lr_psi": 1e-3,
    "replay_capacity": 50000,
    "max_steps": 250,
    "batch_size": 1024,
    "max_episodes": 10000000,
    "gamma": 0.99,
    "entropy_coeff": 0.1,
    "plot_freq": 1000,
    "seed": 42,
}

psi_init = init_psi(NUM_STATES, config["rep_dim"], config["seed"])

print("=== Training BASELINE agent ===")
psi_baseline, metrics_baseline = train(T_baseline_jax, config, psi_init)

In [ ]:
def plot_success_rate(success_list, title, window=100):
    """Plot smoothed success rate."""
    plt.figure(figsize=(8, 4))
    successes = np.array([1.0 if r > 0 else 0.0 for r in success_list])
    smoothed = np.convolve(successes, np.ones(window) / window, mode="valid")
    plt.plot(range(len(smoothed)), smoothed, color="tab:blue", linewidth=2)
    plt.xlabel("Training Episodes", fontsize=12)
    plt.ylabel("Success Rate", fontsize=12)
    plt.title(title, fontsize=13)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

plot_success_rate(metrics_baseline["success_list"], "Baseline Agent — Success Rate")
plt.show()

## Visualization of Training
### Representational similarity of each state to the goal throughout training,
### plus one greedy evaluation trajectory per snapshot.

In [ ]:
def render_maze(ax, sim_map, traj, episode_num, show_absorbing=False):
    """Render maze heatmap with trajectory overlay."""
    ax.clear()
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_xticks([])
    ax.set_yticks([])

    ax.imshow(
        sim_map, cmap="viridis", origin="lower",
        vmin=min(0.0, np.min(sim_map)), vmax=1,
    )
    ax.imshow(
        walls, cmap=plt.cm.binary, vmin=0, vmax=1, alpha=0.5, origin="lower",
    )
    if show_absorbing:
        for xs, ys in absorbing_boundary:
            ax.plot(xs, ys, color="red", linewidth=2.5, solid_capstyle="round")

    traj_coords = np.array(
        [np.unravel_index(s, (HEIGHT, WIDTH)) for s in traj]
    )
    if len(traj_coords) > 1:
        ax.plot(
            traj_coords[:, 1], traj_coords[:, 0],
            color="black", linewidth=2.0, alpha=0.8,
        )

    ax.scatter(
        *np.unravel_index(START_STATE, (HEIGHT, WIDTH))[::-1],
        marker="o", s=200, c="lime", edgecolors="black",
        linewidth=2, zorder=10, clip_on=False,
    )
    ax.scatter(
        *np.unravel_index(GOAL_STATE, (HEIGHT, WIDTH))[::-1],
        marker="*", s=300, c="red", edgecolors="black",
        linewidth=2, zorder=10, clip_on=False,
    )
    ax.set_title(f"Trial {episode_num}", fontsize=24)
    return ax

In [ ]:
# Baseline animation
episodes_b = [h[0] for h in metrics_baseline["similarity_history"]]
sims_b = [h[1] for h in metrics_baseline["similarity_history"]]
trajs_b = [h[2] for h in metrics_baseline["similarity_history"]]

fig, ax = plt.subplots(figsize=(10, 10))
print(f"Total frames: {len(episodes_b)}")

def animate_baseline(idx):
    render_maze(ax, sims_b[idx].reshape((HEIGHT, WIDTH)), trajs_b[idx], episodes_b[idx])

anim = animation.FuncAnimation(
    fig, animate_baseline, frames=len(episodes_b),
    interval=200, blit=False, repeat=True,
)
plt.rcParams["animation.embed_limit"] = 100
anim.save("figures/baseline_animation.gif", writer="pillow", fps=5)
display(HTML(anim.to_jshtml(fps=5, default_mode="loop")))
plt.close(fig)

## Absorbing Failure States Experiment
### Same code, different transition table. The absorbing region (top-right
### quadrant) freezes the agent — all actions loop back to the same state.

In [ ]:
# Visualize absorbing region
fig, ax = plt.subplots(figsize=(6, 6))
display_grid = np.zeros((HEIGHT, WIDTH, 3))
for r in range(HEIGHT):
    for c in range(WIDTH):
        if walls[r, c] == 1:
            display_grid[r, c] = [0.2, 0.2, 0.2]
        else:
            display_grid[r, c] = [1.0, 1.0, 1.0]
ax.imshow(display_grid, origin="lower")
for xs, ys in absorbing_boundary:
    ax.plot(xs, ys, color="red", linewidth=2.5, solid_capstyle="round")
# Label the failure region at its center
abs_rows = [np.unravel_index(s, (HEIGHT, WIDTH))[0] for s in ABSORBING_REGION]
abs_cols = [np.unravel_index(s, (HEIGHT, WIDTH))[1] for s in ABSORBING_REGION]
ax.text(
    np.mean(abs_cols), np.mean(abs_rows), "Failure\nRegion",
    ha="center", va="center", fontsize=11, fontweight="bold",
    color="red", zorder=10,
)
ax.scatter(*GOAL_COORD[::-1], marker="*", s=300, c="red", zorder=5, label="Goal")
ax.scatter(0, 0, marker="o", s=200, c="lime", edgecolors="black", zorder=5, label="Start")
ax.set_title("Four Rooms — Absorbing Failure Region", fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
print("=== Training ABSORBING agent ===")
psi_absorbing, metrics_absorbing = train(T_absorbing_jax, config, psi_init)

plot_success_rate(metrics_absorbing["success_list"], "Absorbing Agent — Success Rate")
plt.show()

In [ ]:
# Absorbing animation
episodes_a = [h[0] for h in metrics_absorbing["similarity_history"]]
sims_a = [h[1] for h in metrics_absorbing["similarity_history"]]
trajs_a = [h[2] for h in metrics_absorbing["similarity_history"]]

fig, ax = plt.subplots(figsize=(10, 10))
print(f"Total frames: {len(episodes_a)}")

def animate_absorbing(idx):
    render_maze(
        ax, sims_a[idx].reshape((HEIGHT, WIDTH)),
        trajs_a[idx], episodes_a[idx], show_absorbing=True,
    )

anim = animation.FuncAnimation(
    fig, animate_absorbing, frames=len(episodes_a),
    interval=200, blit=False, repeat=True,
)
anim.save("figures/absorbing_animation.gif", writer="pillow", fps=5)
display(HTML(anim.to_jshtml(fps=5, default_mode="loop")))
plt.close(fig)

## Comparison: Baseline vs Absorbing

In [ ]:
psi_baseline_np = np.array(psi_baseline)
psi_absorbing_np = np.array(psi_absorbing)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

for ax, psi_np, title, show_abs in [
    (ax1, psi_baseline_np, "Baseline (no absorbing)", False),
    (ax2, psi_absorbing_np, "Absorbing", True),
]:
    goal_vec = psi_np[GOAL_STATE]
    gnorm = np.linalg.norm(goal_vec) + 1e-8
    sim = (psi_np @ goal_vec) / (
        np.linalg.norm(psi_np, axis=1) * gnorm + 1e-8
    )
    sim_map = sim.reshape((HEIGHT, WIDTH))

    im = ax.imshow(
        sim_map, cmap="viridis", origin="lower",
        vmin=min(0, sim.min()), vmax=1,
    )
    ax.imshow(
        walls, cmap=plt.cm.binary, vmin=0, vmax=1, alpha=0.5, origin="lower",
    )
    if show_abs:
        for xs, ys in absorbing_boundary:
            ax.plot(xs, ys, color="red", linewidth=2.5, solid_capstyle="round")
    ax.scatter(
        *np.unravel_index(START_STATE, (HEIGHT, WIDTH))[::-1],
        marker="o", s=200, c="lime", edgecolors="black", linewidth=2, zorder=5,
    )
    ax.scatter(
        *np.unravel_index(GOAL_STATE, (HEIGHT, WIDTH))[::-1],
        marker="*", s=300, c="red", edgecolors="black", linewidth=2, zorder=5,
    )
    ax.set_title(f"{title} — psi(s) . psi(goal)", fontsize=14)
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.tight_layout()
plt.savefig(
    "figures/absorbing_similarity_comparison.png", dpi=150, bbox_inches="tight",
)
plt.show()

# Print average similarities
for psi_np, name in [
    (psi_baseline_np, "Baseline"),
    (psi_absorbing_np, "Absorbing"),
]:
    goal_vec = psi_np[GOAL_STATE]
    gnorm = np.linalg.norm(goal_vec) + 1e-8
    sim = (psi_np @ goal_vec) / (
        np.linalg.norm(psi_np, axis=1) * gnorm + 1e-8
    )
    absorb_sims = [sim[s] for s in ABSORBING_REGION]
    non_absorb_sims = [
        sim[s] for s in range(NUM_STATES)
        if s not in ABSORBING_REGION and walls.flat[s] == 0 and s != GOAL_STATE
    ]
    print(
        f"{name}:  absorbing region avg sim = {np.mean(absorb_sims):.4f},  "
        f"non-absorbing avg sim = {np.mean(non_absorb_sims):.4f}"
    )